# Document embedding / topic modeling experiments
먼저 README의 설치 명령을 저장소 루트에서 실행하세요. 전체 실험은 GPU 환경에서 별도로 검증해야 합니다.


In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Open this notebook from the repository directory")


## 작은 입력 실행
이 결과는 논문 수치 비교에 사용하지 않습니다.


In [ ]:
subprocess.run([sys.executable, "-m", "topic_embedding.cli", "--dataset", "bbc",
    "--model", "sbert", "--pooling", "CX", "--topics", "2", "4",
    "--limit", "200", "--umap-seed", "42"], cwd=ROOT, check=True)


## 가장 최근 결과 읽기


In [ ]:
import json
import pandas as pd

runs = sorted((ROOT / "results").glob("bbc-sbert-*"))
if not runs:
    raise RuntimeError("Run the experiment first")
run = runs[-1]
metadata = json.loads((run / "metadata.json").read_text(encoding="utf-8"))
if metadata["status"] != "complete":
    raise RuntimeError("Latest run did not complete")
metrics = pd.read_csv(run / "metrics.csv")
metrics


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["coherence_c_v", "coherence_c_npmi", "topic_diversity"]):
    for pooling, group in metrics.groupby("pooling"):
        ax.plot(group["nr_topics"], group[metric], marker="o", label=pooling)
    ax.set(xlabel="Requested topics", ylabel=metric)
    ax.legend()
fig.tight_layout()


## 전체 실행 예시
아래 명령은 자동 실행하지 않습니다. 터미널에서 조건을 확인하고 실행하세요.
```bash
topic-experiment --dataset bbc --model sbert --pooling all
topic-experiment --dataset bbc --model llama --pooling all --batch-size 4
```
20ng, imdb도 같은 방식으로 실행합니다. [재현성 문서](../docs/REPRODUCIBILITY.md)를 확인하세요.
